# Tratamento e Limpeza de Dados

### Passo 1 - Leitura e Identificação de Problemas

In [4]:
import pandas as pd 

# Carregar os dados brutos
df = pd.read_csv('~/work-environment/study-files/vendas_brutas.csv')

In [5]:
# 1. Visualizar o DataFrame
display(df)

,id_venda,data,cliente,categoria,valor_venda,quantidade,regiao
0,1001,2024-01-15,Ana Silva,Eletrônicos,1200.5,1.0,Sudeste
1,1002,2024-01-16,BRUNO SOUZA,Móveis,850.0,2.0,Sul
2,1002,2024-01-16,BRUNO SOUZA,Móveis,850.0,2.0,Sul
3,1003,2024-01-16,Carla Dias,Eletrônicos,NaN,1.0,Sudeste
4,1004,2024-01-17,Diego Alves,Vestuário,150.0,NaN,Nordeste
5,1005,2024-01-18,Elena Lima,Móveis,1200.0,1.0,Centro-Oeste
6,1006,2024-01-18,fabio costa,Eletrônicos,250.0,2.0,NaN
7,1007,2024-01-19,Gisele Rocha,Vestuário,450.0,4.0,Sul


In [6]:
# 2. Verificar valores nulos / faltantes por coluna
display(df.isna().sum())

id_venda       0
data           0
cliente        0
categoria      0
valor_venda    1
quantidade     1
regiao         1
dtype: int64

Dessa forma já consigo identificar a quantidade de nulos que tem no dataset de uma vez, entendendo o que já posso fazer.

### Passo 2 - Lidando com Nulos e Duplicatas

In [7]:
display(df)

,id_venda,data,cliente,categoria,valor_venda,quantidade,regiao
0,1001,2024-01-15,Ana Silva,Eletrônicos,1200.5,1.0,Sudeste
1,1002,2024-01-16,BRUNO SOUZA,Móveis,850.0,2.0,Sul
2,1002,2024-01-16,BRUNO SOUZA,Móveis,850.0,2.0,Sul
3,1003,2024-01-16,Carla Dias,Eletrônicos,NaN,1.0,Sudeste
4,1004,2024-01-17,Diego Alves,Vestuário,150.0,NaN,Nordeste
5,1005,2024-01-18,Elena Lima,Móveis,1200.0,1.0,Centro-Oeste
6,1006,2024-01-18,fabio costa,Eletrônicos,250.0,2.0,NaN
7,1007,2024-01-19,Gisele Rocha,Vestuário,450.0,4.0,Sul


In [10]:
# 1. Remover linhas duplicadas mantendo a primeira ocorrência
df = df.drop_duplicates()

# 2. Preencher nulos com estratégias específicas por coluna
df['valor_venda'] = df['valor_venda'].fillna(df['valor_venda'].median())

df['quantidade'] = df['quantidade'].fillna(1)

df['regiao'] = df['regiao'].fillna('Não Informada')

display(df.isna().sum())

display(df)

id_venda       0
data           0
cliente        0
categoria      0
valor_venda    0
quantidade     0
regiao         0
dtype: int64

,id_venda,data,cliente,categoria,valor_venda,quantidade,regiao
0,1001,2024-01-15,Ana Silva,Eletrônicos,1200.5,1.0,Sudeste
1,1002,2024-01-16,BRUNO SOUZA,Móveis,850.0,2.0,Sul
3,1003,2024-01-16,Carla Dias,Eletrônicos,650.0,1.0,Sudeste
4,1004,2024-01-17,Diego Alves,Vestuário,150.0,1.0,Nordeste
5,1005,2024-01-18,Elena Lima,Móveis,1200.0,1.0,Centro-Oeste
6,1006,2024-01-18,fabio costa,Eletrônicos,250.0,2.0,Não Informada
7,1007,2024-01-19,Gisele Rocha,Vestuário,450.0,4.0,Sul


O *.fillna()* faz o procedimento de forma rápida e eficiente, sendo vetorizado. Quando colocamos .fillna(1) na coluna quantidade, ele varre todas as linhas que na coluna quantidade estiverem nulos, e coloca o número 1. Da mesma forma ele faz com .median() e com o .fillna('Nâo Informado'). A questão é que ele resolve esses problemas de uma vez.

**Como identificar duplicatas se a tabela estiver com 1 milhão de linhas?**

Usamos o método *.duplicated()*. Ele faz uma varredura vatorizada em todo o DataFrame de 1 milhão de linhas e retorna uma Series de True / False indicando quais linhas já apareceram antes. também podemos fazer uma contagem, assim fica mais fácil ver quantas linhas existem:

In [11]:
total_duplicatas = df.duplicated().sum()

display(f"Total de linhas duplicadas: {total_duplicatas}")

'Total de linhas duplicadas: 0'

Nesse cenário deu zero por que já corrigimos as duplicatas anteriormente.

### Passo 3 - Convenção de Tipos e Dtypes Nullable

Para garantir que cada coluna esteja com seu tipo certo, no Pandas, o *dtype* é fundamental para economizar memória e evitar erros em cálculos ou modelos de machine learning.

1 - As ferramentas de conversão tradicionais

- *.astype()* : Converte o tipo de uma coluna inteira para outro (ex: converter texto para float, ou float para inteiro).

- *pd.to_datatime()* : Converte colunas de texto com datas (ex: "2024-01-15") para o formato nativo de data do Pandas (datetime64), permitindo extrair anos, meses, dias da semana, e filtrar por intervalos de tempo.

- *pd.to_numeric()* : Converte colunas numéricas que venham com sujeiras ou texto para números reais, permitindo tratar erros com parâmetro *erros='coerce'* (tranformando o que for inválido em NaN)

2 - Tipos modernos (Nullable Dtypes)

- *Int64* : Inteiro que aceita valores nulos sem virar float

- *boolean* : Booleano que aceita True, False e ausência de dados (<NA>)

- *category* : Categoria, ideal para colunas com textos que se repetem muito (como 'regiao' ou 'categoria'), pois compacta o uso da memória drástucamente.



In [12]:
display(df.dtypes)

id_venda         int64
data               str
cliente            str
categoria          str
valor_venda    float64
quantidade     float64
regiao             str
dtype: object

In [14]:
# 1. Convertendo a coluna 'data' para o tipo datatime real do Pandas
df['data'] = pd.to_datetime(df['data'])

# 2. Convertendo 'quantidade' para o tipo Nullable Integer (Int64)
df['quantidade'] = df['quantidade'].astype('Int64')

# 3. Convertendo 'regiao' e 'categoria' para o tipo Categoria (otimização de memória)
df['regiao'] = df['regiao'].astype('category')

df['categoria'] = df['categoria'].astype('category')

# 4. Conferir os novos dtypes da tabela
display(df.dtypes)
display(df)

id_venda                int64
data           datetime64[us]
cliente                   str
categoria            category
valor_venda           float64
quantidade              Int64
regiao               category
dtype: object

,id_venda,data,cliente,categoria,valor_venda,quantidade,regiao
0,1001,2024-01-15,Ana Silva,Eletrônicos,1200.5,1,Sudeste
1,1002,2024-01-16,BRUNO SOUZA,Móveis,850.0,2,Sul
3,1003,2024-01-16,Carla Dias,Eletrônicos,650.0,1,Sudeste
4,1004,2024-01-17,Diego Alves,Vestuário,150.0,1,Nordeste
5,1005,2024-01-18,Elena Lima,Móveis,1200.0,1,Centro-Oeste
6,1006,2024-01-18,fabio costa,Eletrônicos,250.0,2,Não Informada
7,1007,2024-01-19,Gisele Rocha,Vestuário,450.0,4,Sul


### Passo 4 - Manipulação de Texto e Funções de Usuário

In [15]:
display(df)

,id_venda,data,cliente,categoria,valor_venda,quantidade,regiao
0,1001,2024-01-15,Ana Silva,Eletrônicos,1200.5,1,Sudeste
1,1002,2024-01-16,BRUNO SOUZA,Móveis,850.0,2,Sul
3,1003,2024-01-16,Carla Dias,Eletrônicos,650.0,1,Sudeste
4,1004,2024-01-17,Diego Alves,Vestuário,150.0,1,Nordeste
5,1005,2024-01-18,Elena Lima,Móveis,1200.0,1,Centro-Oeste
6,1006,2024-01-18,fabio costa,Eletrônicos,250.0,2,Não Informada
7,1007,2024-01-19,Gisele Rocha,Vestuário,450.0,4,Sul


In [17]:
# 1. Remover espaços extras e padronizar o nome dos clientes (ex: primeira letra maiúscula)
df['cliente'] = df['cliente'].str.strip().str.title()

# 2. Conferir o resultado da coluna cliente limpa
display(df[['id_venda', 'cliente', 'regiao']])

,id_venda,cliente,regiao
0,1001,Ana Silva,Sudeste
1,1002,Bruno Souza,Sul
3,1003,Carla Dias,Sudeste
4,1004,Diego Alves,Nordeste
5,1005,Elena Lima,Centro-Oeste
6,1006,Fabio Costa,Não Informada
7,1007,Gisele Rocha,Sul


### Passo 5 - Avançando para funções de Usuários com *.apply()*

Vamos usar uma hipótese de que precisamos criar uma regra de negócio: Classificar o porte de cada venda com base no valor_venda

- Vendas acima de R$ 1.000,00 = 'Alto Padrão'
- Vendas abaixo ou igual a R$ 1.000,00 = 'Padrão'

Para fazer isso de forma customizada, vamos criar uma função em python e aplicamos no DataFrame usando *.apply()*

In [19]:
# 1. Definir uma função de usuário (UDF) para classificar a venda
def classificacao_vendas(valor):
    if valor > 1000:
        return 'Alto Padrão'
    else:
        return 'Padrão'

# 2. Aplicar a função linha por linha na coluna 'valor_venda' usando .apply()
df['classificacao'] = df['valor_venda'].apply(classificacao_vendas)

# 3. Visualizar o resultado com as colunas principais
display(df[['id_venda', 'cliente', 'valor_venda', 'classificacao']])

,id_venda,cliente,valor_venda,classificacao
0,1001,Ana Silva,1200.5,Alto Padrão
1,1002,Bruno Souza,850.0,Padrão
3,1003,Carla Dias,650.0,Padrão
4,1004,Diego Alves,150.0,Padrão
5,1005,Elena Lima,1200.0,Alto Padrão
6,1006,Fabio Costa,250.0,Padrão
7,1007,Gisele Rocha,450.0,Padrão


### O que aprendi

Nessas consultas, eu entendi a importância de manipular e mexer nos dados quando são recebidos. Vendo assim, posso organizar o tratamento sendo uma das primeiras coisas a se fazer quando recebe os dados, para tratar, corrigir, modificar e criar o dataset limpo, e assim, após isso fazer uso de um dataset limpo. para análises exploratórias e recursos de machine learning, essa rotina vai ajudar bastante.